In [ ]:
#Importer les bbliotheque necessaire
#Pytorch et pyplot de matplotlib 

import torch
import torchvision
import matplotlib.pyplot as pyplot

print("Pytorch :", torch.__version__)
print("torchvision : ", torchvision.__version__)

Crée une transformation qui convertit chaque image en tenseur PyTorch. Les pixels passent de valeurs entières entre 0 et 255 à des nombres décimaux entre 0 et 1. Le modèle apprend généralement plus facilement avec ces valeurs normalisées.

In [ ]:
#Crée une transformation qui convertit chaque image en tenseur PyTorch
#Cette transformation sera appliquée à chaque image du dataset lors du chargement
#Elle permet de convertir les images en tenseurs PyTorch, ce qui est nécessaire pour les utiliser avec les modèles PyTorch
transform = torchvision.transforms.ToTensor()

Charge les images qui serviront à entraîner le modèle.

root="data" : indique où enregistrer les fichiers du jeu de données.
train=True : demande la partie entraînement.
download=True : télécharge MNIST si les fichiers ne sont pas déjà présents.
transform=transform : applique la conversion en tenseurs à chaque image.

In [ ]:

#Télécharge et charge le dataset MNIST pour l'entraînement
#Le paramètre 'root' spécifie le répertoire où les données seront stockées
#Le paramètre 'train' indique que nous voulons le jeu de données d'entraînement
#Le paramètre 'download' permet de télécharger les données si elles ne sont pas déjà présentes
#Le paramètre 'transform' applique la transformation définie précédemment à chaque image
train_data = torchvision.datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

Charge la partie test, qui servira plus tard à vérifier si le modèle reconnaît aussi des images qu’il n’a pas utilisées pour apprendre. train=False sélectionne ces données.

In [ ]:
#Télécharge et charge le dataset MNIST pour le test
#Le paramètre 'root' spécifie le répertoire où les données seront stockées
#Le paramètre 'train' indique que nous voulons le jeu de données de test
#Le paramètre 'download' permet de télécharger les données si elles ne sont pas déjà présentes
#Le paramètre 'transform' applique la transformation définie précédemment à chaque image
test_data = torchvision.datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

len(...) compte le nombre d’images dans chaque jeu de données. Le f devant le texte permet d’insérer le résultat directement dans la chaîne avec {...}.

In [ ]:
print(f"Images d'entraînement : {len(train_data)}")
print(f"Images de test : {len(test_data)}")



In [ ]:
#Afficher un exemple d'image d'entraînement
import matplotlib.pyplot as plt
image, label = train_data[0]
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label : {label}")
plt.show()



In [ ]:
#Afficher un exemple d'image de test
image, label = test_data[0]
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Label : {label}")
plt.show()

Récupère le premier exemple d’entraînement. Chaque exemple contient deux éléments :

image : l’image du chiffre, sous forme de tenseur ;
label : le chiffre attendu, un nombre entre 0 et 9.

In [ ]:
image, label = train_data[0]

Affiche la forme du tenseur et l’étiquette. La forme devrait ressembler à torch.Size([1, 28, 28]) : 1 canal (niveaux de gris), 28 pixels de haut et 28 pixels de large.

In [ ]:
print(f"Forme de l'image : {image.shape}")
print(f"Chiffre attendu : {label}")

Affiche l’image :

squeeze() retire la dimension du canal, qui vaut 1, pour que l’affichage reçoive une matrice de 28 × 28 pixels ;
cmap="gray" choisit une palette en niveaux de gris.

In [ ]:
pyplot.imshow(image.squeeze(), cmap="gray")

Ajoute le chiffre attendu comme titre, masque les axes, puis affiche le graphique.

In [ ]:
pyplot.title(f"Chiffre : {label}")
pyplot.axis("off")
pyplot.show()

In [ ]:
#importe l’outil PyTorch qui organise un jeu de données en lots.
from torch.utils.data import DataLoader

batch_size = 64             #Le modèle en traitera jusqu’à 64 avant de mettre à jour ses paramètres.

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True            #Mélange les données d'entraînement à chaque époque pour améliorer l'apprentissage.
)

test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False           #Ne mélange pas les données de test pour conserver l'ordre original.
)

images, labels = next(iter(train_loader))

print("Forme des images :", images.shape)
print("Forme des étiquettes :", labels.shape)
print("Quelques étiquettes :", labels[:10])

In [ ]:
#importe les outils PyTorch pour construire des réseaux. nn est simplement un raccourci pratique.
import torch.nn as nn

model = nn.Sequential(          #crée un réseau qui applique les couches dans l’ordre où elles sont écrites.
    nn.Flatten(),               #transforme chaque image de forme [1, 28, 28] en une liste de 784 pixels. Pour le lot, la forme devient [64, 784].
    nn.Linear(28 * 28, 128),    #relie les 784 pixels à 128 neurones.784 entrées vers 128 neurones
    nn.ReLU(),                  #activation des 128 valeurs
    nn.Linear(128, 10)          #128 valeurs vers 10 scores
)

scores = model(images)          #fait passer le lot d’images dans le réseau.

print("Forme des scores :", scores.shape)
print("Scores de la première image :", scores[0])

Préparer deux outils indispensables pour l’entraînement : une mesure de l’erreur et un mécanisme pour ajuster le modèle.

In [ ]:
criterion = nn.CrossEntropyLoss()   #crée une fonction qui mesure l’écart entre les 10 scores produits par le modèle et le chiffre attendu (labels)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) #crée l’optimiseur, chargé d’ajuster les paramètres du réseau pour réduire cette perte.

scores = model(images)              #pour obtenir ses 10 scores par image
loss = criterion(scores, labels)    #calcule la perte entre les scores prédits et les étiquettes réelles

print("Perte initiale :", loss.item())

faisons une seule mise à jour des paramètres avec un lot d’images.

In [ ]:
#efface les gradients calculés précédemment. PyTorch les accumule par défaut ; on les remet donc à zéro avant ce nouvel apprentissage.
optimizer.zero_grad()

scores = model(images)
loss_before = criterion(scores, labels)

loss_before.backward()      #calcule les gradients : ils indiquent comment les paramètres du modèle contribuent à l’erreur.
optimizer.step()            #met à jour les paramètres du modèle en utilisant les gradients calculés.

with torch.no_grad():
    scores_after = model(images)
    loss_after = criterion(scores_after, labels)

print(f"Perte avant la mise à jour : {loss_before.item():.4f}")
print(f"Perte après la mise à jour : {loss_after.item():.4f}")

Maintenant, faisons une époque complète, c’est-à-dire un passage sur les 60 000 images d’entraînement, réparties en lots.

In [ ]:
total_loss = 0.0
total_correct = 0
total_images = 0

for images, labels in train_loader:
    optimizer.zero_grad()

    scores = model(images)
    loss = criterion(scores, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item() * labels.size(0)

    predictions = scores.argmax(dim=1)        #choisit, pour chaque image, l’indice du score le plus élevé
    total_correct += (predictions == labels).sum().item()   #compte combien de prédictions sont correctes
    total_images += labels.size(0)                          #compte le nombre total d’images traitées

average_loss = total_loss / total_images
accuracy = total_correct / total_images

print(f"Perte moyenne d'entraînement : {average_loss:.4f}")
print(f"Exactitude pendant l'entraînement : {accuracy:.1%}")

Test

In [ ]:
model.eval()

test_loss = 0.0
total_correct = 0
total_images = 0

with torch.no_grad():
    for images, labels in test_loader:
        scores = model(images)
        loss = criterion(scores, labels)

        test_loss += loss.item() * labels.size(0)

        predictions = scores.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_images += labels.size(0)

average_test_loss = test_loss / total_images
test_accuracy = total_correct / total_images

print(f"Perte moyenne sur les données de test : {average_test_loss:.4f}")
print(f"Exactitude sur les données de test : {test_accuracy:.1%}")

model.train()

Le modèle reconnaît la plupart des chiffres, mais il fait encore des erreurs. Regardons quelques-unes pour voir quels chiffres il confond.

In [ ]:
model.eval()

wrong_images = []
wrong_labels = []
wrong_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        scores = model(images)
        predictions = scores.argmax(dim=1)

        wrong_indices = (predictions != labels).nonzero(as_tuple=True)[0]

        for index in wrong_indices.tolist()[:8 - len(wrong_images)]:
            wrong_images.append(images[index].squeeze().cpu())
            wrong_labels.append(labels[index].item())
            wrong_predictions.append(predictions[index].item())

        if len(wrong_images) == 8:
            break

if wrong_images:
    figure, axes = pyplot.subplots(2, 4, figsize=(10, 5))

    for axis, image, expected, predicted in zip(
        axes.flat, wrong_images, wrong_labels, wrong_predictions
    ):
        axis.imshow(image, cmap="gray")
        axis.set_title(f"Réponse : {expected} | Prédit : {predicted}")
        axis.axis("off")

    pyplot.tight_layout()
    pyplot.show()
else:
    print("Aucune erreur trouvée.")

model.train()

regardons maintenant quels chiffres sont le plus souvent confondus grâce à une matrice de confusion.

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        scores = model(images)
        predictions = scores.argmax(dim=1)

        for expected, predicted in zip(labels, predictions):
            confusion[expected, predicted] += 1

pyplot.figure(figsize=(8, 6))
pyplot.imshow(confusion, cmap="Blues")
pyplot.colorbar()
pyplot.xlabel("Chiffre prédit")
pyplot.ylabel("Vrai chiffre")
pyplot.title("Matrice de confusion sur les données de test")

for row in range(10):
    for column in range(10):
        pyplot.text(column, row, int(confusion[row, column]),
                    ha="center", va="center")

pyplot.xticks(range(10))
pyplot.yticks(range(10))
pyplot.show()

model.train()